In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── sector risk parameters ────────────────────────────────────
SECTOR_RISK = {
    "Technology":            {"base_loss_rate": 0.10, "severity": 180_000, "primary_line": "Tech & AI Liability"},
    "Financial Services":    {"base_loss_rate": 0.12, "severity": 220_000, "primary_line": "Cyber Liability + D&O"},
    "Healthcare":            {"base_loss_rate": 0.11, "severity": 250_000, "primary_line": "Tech & AI Liability + Cyber"},
    "Communication Services":{"base_loss_rate": 0.09, "severity": 160_000, "primary_line": "Cyber Liability"},
    "Consumer Cyclical":     {"base_loss_rate": 0.07, "severity": 130_000, "primary_line": "CGL"},
    "Industrials":           {"base_loss_rate": 0.07, "severity": 120_000, "primary_line": "CGL"},
    "Other":                 {"base_loss_rate": 0.08, "severity": 150_000, "primary_line": "Tech & AI Liability"},
}

def get_sector_params(sector):
    return SECTOR_RISK.get(sector, SECTOR_RISK["Other"])

def risk_label(score):
    if score < 0.30: return "low"
    if score < 0.50: return "medium"
    if score < 0.70: return "high"
    return "critical"

# ── fetch company data ────────────────────────────────────────
def fetch_company(ticker_str):
    t    = yf.Ticker(ticker_str)
    info = t.info
    hist = t.history(period="1y")
    return info, hist

def compute_risk_score(info, hist):
    scores = {}

    # 1. volatility
    if len(hist) > 20:
        daily_vol = hist['Close'].pct_change().dropna().std()
        ann_vol   = daily_vol * np.sqrt(252)
        scores['volatility'] = min(ann_vol / 0.6, 1.0)
    else:
        scores['volatility'] = 0.5

    # 2. beta
    beta = info.get('beta', 1.0) or 1.0
    scores['beta'] = min(abs(beta) / 3.0, 1.0)

    # 3. leverage
    dte = info.get('debtToEquity', 50) or 50
    scores['leverage'] = min(dte / 300, 1.0)

    # 4. governance
    audit = info.get('auditRisk', 5) or 5
    board = info.get('boardRisk', 5) or 5
    scores['governance'] = min((audit + board) / 20, 1.0)

    # 5. growth stress
    rev_growth = info.get('revenueGrowth', 0.05) or 0.05
    scores['growth_stress'] = max(0, min(-rev_growth + 0.1, 1.0))

    weights = {
        'volatility':    0.30,
        'beta':          0.20,
        'leverage':      0.20,
        'governance':    0.20,
        'growth_stress': 0.10,
    }
    composite = sum(scores[k] * weights[k] for k in weights)
    return composite, scores

def estimate_premium(info, risk_score):
    market_cap = info.get('marketCap', 100_000_000) or 100_000_000
    base       = np.log10(max(market_cap, 1e6)) * 3_000
    loading    = 1 + (risk_score * 2)
    premium    = base * loading
    return round(premium / 1000) * 1000

def print_result(ticker_str):
    print(f"\nfetching data for {ticker_str.upper()}...")
    info, hist       = fetch_company(ticker_str.upper())
    risk_score, subs = compute_risk_score(info, hist)
    sector           = info.get('sector', 'Other')
    sector_params    = get_sector_params(sector)
    premium          = estimate_premium(info, risk_score)
    label            = risk_label(risk_score)
    name             = info.get('shortName', ticker_str.upper())
    market_cap       = info.get('marketCap', 0) or 0

    print(f"\n── {name} ({'  '.join([ticker_str.upper(), sector])}) ──────────────────")
    print(f"  risk score       : {risk_score:.3f}  ({label})")
    print(f"  est. premium     : ${premium:,.0f}/yr")
    print(f"  primary line     : {sector_params['primary_line']}")
    print(f"  market cap       : ${market_cap/1e9:.1f}B")
    print()
    print("  risk factor breakdown:")
    for factor, score in subs.items():
        bar   = '█' * int(score * 20)
        print(f"    {factor:<16} {score:.2f}  {bar}")
    print()

    if label == 'low':
        print("  recommendation: bind at standard terms")
    elif label == 'medium':
        print("  recommendation: bind with standard exclusions and cyber sublimit")
    elif label == 'high':
        print("  recommendation: elevated terms — 20% premium loading required")
    else:
        print("  recommendation: refer to senior underwriter — do not bind without review")

    print()
    print("  coverage questions:")
    print("  • does the company use AI or ML in its core product?")
    print("  • what is the data retention and encryption policy?")
    print("  • any regulatory actions or SEC inquiries in last 24 months?")
    print("  • what % of revenue comes from a single customer?")
    print("  • is the company profitable or burning cash?")

    # ── charts ────────────────────────────────────────────────
    plt.rcParams.update({
        'figure.facecolor': '#0f0f0e', 'axes.facecolor':  '#0f0f0e',
        'axes.edgecolor':   '#333332', 'axes.labelcolor': '#ccccca',
        'xtick.color':      '#666664', 'ytick.color':     '#666664',
        'text.color':       '#ccccca', 'grid.color':      '#222221',
        'grid.linewidth':   0.5,       'axes.grid':       True,
        'font.family':      'monospace',
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor('#0f0f0e')

    # price chart
    axes[0].plot(hist.index, hist['Close'], color='#378ADD', linewidth=1.5)
    axes[0].fill_between(hist.index, hist['Close'], alpha=0.08, color='#378ADD')
    axes[0].set_title(f'{name.upper()} — CLOSE PRICE (1Y)',
                      fontsize=10, color='#888886', pad=8)

    # risk factor bar chart
    factors = list(subs.keys())
    values  = [subs[f] for f in factors]
    colors  = ['#E24B4A' if v > 0.6 else '#EF9F27' if v > 0.3 else '#1D9E75' for v in values]
    axes[1].barh(factors, values, color=colors, alpha=0.85)
    axes[1].set_xlim(0, 1)
    axes[1].axvline(0.5, color='#444442', linewidth=1, linestyle=':')
    axes[1].set_title('RISK FACTOR BREAKDOWN', fontsize=10, color='#888886', pad=8)

    plt.tight_layout()
    plt.savefig(f'{ticker_str.upper()}_risk.png', dpi=150, facecolor='#0f0f0e')
    plt.show()


# ── run ───────────────────────────────────────────────────────
tickers = ['CRWD', 'SNOW', 'OKTA']

for t in tickers:
    print_result(t)


fetching data for CRWD...

── CrowdStrike Holdings, Inc. (CRWD  Technology) ──────────────────
  risk score       : 0.510  (high)
  est. premium     : $68,000/yr
  primary line     : Tech & AI Liability
  market cap       : $170.8B

  risk factor breakdown:
    volatility       0.75  ███████████████
    beta             0.41  ████████
    leverage         0.06  █
    governance       0.95  ███████████████████
    growth_stress    0.00  

  recommendation: elevated terms — 20% premium loading required

  coverage questions:
  • does the company use AI or ML in its core product?
  • what is the data retention and encryption policy?
  • any regulatory actions or SEC inquiries in last 24 months?
  • what % of revenue comes from a single customer?
  • is the company profitable or burning cash?

fetching data for SNOW...

── Snowflake Inc. (SNOW  Technology) ──────────────────
  risk score       : 0.626  (high)
  est. premium     : $74,000/yr
  primary line     : Tech & AI Liability
  marke